# 🎛️ TIL Metric Scenario Lab

### Mexa no cenário. Observe as métricas. Explique o que aconteceu.

Este laboratório complementa a **Aula 13 — Métricas e Indicadores**.

A regra mais importante é: **você não move Accuracy, Precision, Recall ou F1 diretamente**. Você altera as condições do problema — como `threshold`, prevalência e custos — e as métricas são recalculadas a partir das decisões simuladas.

Isso evita combinações matematicamente impossíveis e aproxima o exercício de uma situação real.


## 📘 Glossário Vivo — sua bússola durante o laboratório

Este laboratório foi desenhado para ser usado **junto com o Glossário Vivo do TIL**. Não tente memorizar todos os conceitos antes de começar: mova os controles, observe o que mudou e consulte o termo correspondente.

### Palavras-chave essenciais

**[Matriz de confusão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#matriz-de-confusão) · [Acurácia](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#acurácia) · [Precisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#precisão) · [Recall](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#recall) · [F1-score](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#f1-score) · [Falso positivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-positivo) · [Falso negativo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-negativo) · [Especificidade](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#especificidade) · [Balanced Accuracy](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#balanced-accuracy) · [Support](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#support) · [Threshold de decisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#threshold-de-decisão) · [Taxa de abstenção](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-abstenção)**

### Como usar o glossário aqui

| Se você observar... | Consulte primeiro... | Pergunta para fazer a si mesmo |
|---|---|---|
| muitos positivos escapando | [Recall](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#recall) e [Falso negativo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-negativo) | Quantos positivos reais estou deixando passar? |
| muitos alertas incorretos | [Precisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#precisão) e [Falso positivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-positivo) | Posso confiar nos positivos que o sistema sinaliza? |
| acurácia muito alta em classe rara | [Acurácia](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#acurácia) e [Balanced Accuracy](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#balanced-accuracy) | A classe majoritária está mascarando o problema? |
| mudança brusca ao mover o corte | [Threshold de decisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#threshold-de-decisão) | Estou mudando o modelo ou apenas a regra de decisão? |
| dúvida sobre TP, FP, FN e TN | [Matriz de confusão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#matriz-de-confusão) | Em qual quadrante o erro apareceu? |
| classes muito desbalanceadas | [Support](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#support) | Quantos exemplos reais existem em cada classe? |

➡️ [Abrir o Glossário Vivo completo — PT-BR](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md)  
➡️ [Open the Living Glossary — EN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.en.md)

> **Estratégia de estudo:** mexa → observe → interprete → consulte o conceito → mexa novamente.


## O que você poderá manipular

- **Cenário**: fraude, triagem médica, spam ou reclamação crítica.
- **[Threshold](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#threshold-de-decisão)**: quão exigente o sistema é antes de classificar algo como positivo.
- **Prevalência**: proporção real de positivos no universo analisado; observe como ela afeta [Acurácia](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#acurácia), [Precisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#precisão) e o [Support](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#support).
- **Separação do modelo**: abstração didática da capacidade de distinguir positivos e negativos.
- **Custos**: falso positivo, falso negativo, revisão humana e inferência.

Enquanto você altera os controles, observe simultaneamente: `Accuracy · Precision · Recall · F1 · Specificity · Balanced Accuracy · FP · FN · custo esperado`.

> O laboratório usa `ipywidgets`, NumPy, Pandas e Matplotlib e foi pensado para executar com **Internet OFF**. Os links do Glossário exigem conexão apenas se você decidir abri-los no GitHub; o simulador em si permanece totalmente offline.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

SEED = 42
N_CASES = 2000
rng = np.random.default_rng(SEED)
BASE_POS = rng.normal(0, 1, N_CASES)
BASE_NEG = rng.normal(0, 1, N_CASES)

print('Ambiente preparado.')
print('Seed:', SEED)
print('Casos simulados:', N_CASES)


In [ ]:
GLOSSARY = 'https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md'

GLOSSARY_LINKS = {
    'accuracy': f'{GLOSSARY}#acurácia',
    'precision': f'{GLOSSARY}#precisão',
    'recall': f'{GLOSSARY}#recall',
    'f1': f'{GLOSSARY}#f1-score',
    'specificity': f'{GLOSSARY}#especificidade',
    'balanced_accuracy': f'{GLOSSARY}#balanced-accuracy',
    'fp': f'{GLOSSARY}#falso-positivo',
    'fn': f'{GLOSSARY}#falso-negativo',
    'confusion_matrix': f'{GLOSSARY}#matriz-de-confusão',
    'threshold': f'{GLOSSARY}#threshold-de-decisão',
    'support': f'{GLOSSARY}#support',
}

def glink(key, label):
    return f"<a href='{GLOSSARY_LINKS[key]}' target='_blank'><b>{label}</b></a>"

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def safe_div(num, den):
    return num / den if den else 0.0

def simulate(prevalence, separation, threshold):
    n_pos = max(1, int(round(N_CASES * prevalence)))
    n_neg = N_CASES - n_pos
    pos_scores = sigmoid(BASE_POS[:n_pos] + separation)
    neg_scores = sigmoid(BASE_NEG[:n_neg] - separation)
    y_true = np.concatenate([np.ones(n_pos, dtype=int), np.zeros(n_neg, dtype=int)])
    scores = np.concatenate([pos_scores, neg_scores])
    y_pred = (scores >= threshold).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    return {'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn}

def metrics_from_cm(cm):
    tp, fp, fn, tn = cm['TP'], cm['FP'], cm['FN'], cm['TN']
    accuracy = safe_div(tp + tn, tp + fp + fn + tn)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    f1 = safe_div(2 * precision * recall, precision + recall)
    balanced_accuracy = (recall + specificity) / 2
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1': f1, 'Specificity': specificity, 'Balanced Accuracy': balanced_accuracy}

def expected_cost(cm, cost_fp, cost_fn, review_cost, inference_cost):
    predicted_positive = cm['TP'] + cm['FP']
    return cm['FP'] * cost_fp + cm['FN'] * cost_fn + predicted_positive * review_cost + N_CASES * inference_cost

def metric_bar(label, value):
    pct = max(0, min(100, value * 100))
    glossary_key = {'Accuracy':'accuracy','Precision':'precision','Recall':'recall','F1':'f1','Specificity':'specificity','Balanced Accuracy':'balanced_accuracy'}[label]
    linked_label = glink(glossary_key, label)
    return f'''<div style="margin:8px 0"><div style="display:flex;justify-content:space-between;font-weight:600"><span>{linked_label}</span><span>{value:.3f}</span></div><div style="height:13px;background:#e9ecef;border-radius:8px;overflow:hidden"><div style="width:{pct:.1f}%;height:100%;background:#4a77b5"></div></div></div>'''

def interpret(metrics, cm, prevalence, threshold, cost_fp, cost_fn, total_cost):
    precision = metrics['Precision']
    recall = metrics['Recall']
    notes = []
    concepts = set()
    if prevalence < 0.10 and metrics['Accuracy'] > 0.90:
        notes.append('A classe positiva é rara e a acurácia é alta. Não use a acurácia isoladamente.')
        concepts.update(['accuracy','balanced_accuracy','support'])
    if recall >= 0.85:
        notes.append('Recall alto: poucos positivos reais estão escapando.')
        concepts.add('recall')
    elif recall < 0.60:
        notes.append('Recall baixo: há muitos falsos negativos em relação aos positivos reais.')
        concepts.update(['recall','fn'])
    if precision >= 0.85:
        notes.append('Precision alta: os alertas positivos tendem a ser confiáveis.')
        concepts.add('precision')
    elif precision < 0.60:
        notes.append('Precision baixa: uma parcela relevante dos alertas positivos é falsa.')
        concepts.update(['precision','fp'])
    if cost_fn >= 5 * max(cost_fp, 0.01):
        notes.append('FN é muito mais caro que FP; um threshold mais baixo pode ser justificável para ganhar Recall.')
        concepts.update(['fn','threshold','recall'])
    elif cost_fp >= 5 * max(cost_fn, 0.01):
        notes.append('FP é muito mais caro; um threshold mais alto pode ser justificável para reduzir falsos alertas.')
        concepts.update(['fp','threshold','precision'])
    if threshold <= 0.35:
        notes.append('Threshold permissivo: tende a aumentar Recall e também os falsos positivos.')
        concepts.update(['threshold','recall','fp'])
    elif threshold >= 0.70:
        notes.append('Threshold exigente: tende a reduzir FP, mas pode aumentar FN.')
        concepts.update(['threshold','fp','fn'])
    notes.append(f"FP={cm['FP']}, FN={cm['FN']}, F1={metrics['F1']:.3f}, custo estimado=R$ {total_cost:,.2f}.")
    concepts.update(['confusion_matrix','f1'])
    labels = {'accuracy':'Acurácia','precision':'Precisão','recall':'Recall','f1':'F1-score','specificity':'Especificidade','balanced_accuracy':'Balanced Accuracy','fp':'Falso positivo','fn':'Falso negativo','confusion_matrix':'Matriz de confusão','threshold':'Threshold','support':'Support'}
    glossary_html = ' · '.join(glink(key, labels[key]) for key in sorted(concepts))
    explanation = '<br><br>'.join('• ' + note for note in notes)
    return explanation + f"<hr><b>📘 Glossário recomendado para este cenário:</b><br>{glossary_html}"


In [ ]:
SCENARIOS = {
    'Fraude bancária': dict(prevalence=.05, separation=1.15, threshold=.50, cost_fp=10., cost_fn=250., review_cost=4., inference_cost=.02),
    'Triagem médica': dict(prevalence=.12, separation=1.00, threshold=.40, cost_fp=25., cost_fn=500., review_cost=18., inference_cost=.03),
    'Filtro de spam': dict(prevalence=.30, separation=1.25, threshold=.60, cost_fp=60., cost_fn=3., review_cost=0., inference_cost=.005),
    'Reclamação crítica': dict(prevalence=.08, separation=.95, threshold=.45, cost_fp=8., cost_fn=120., review_cost=7., inference_cost=.01),
}

class MetricScenarioLab:
    def __init__(self):
        self.scenario = widgets.Dropdown(options=list(SCENARIOS), value='Fraude bancária', description='Cenário:', layout=widgets.Layout(width='420px'))
        self.threshold = widgets.FloatSlider(min=.05, max=.95, step=.05, value=.50, description='Threshold:', readout_format='.2f', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.prevalence = widgets.FloatSlider(min=.01, max=.50, step=.01, value=.05, description='Prevalência:', readout_format='.0%', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.separation = widgets.FloatSlider(min=.30, max=2.50, step=.05, value=1.15, description='Separação:', readout_format='.2f', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.cost_fp = widgets.FloatSlider(min=0, max=300, step=5, value=10, description='Custo FP:', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.cost_fn = widgets.FloatSlider(min=0, max=800, step=10, value=250, description='Custo FN:', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.review_cost = widgets.FloatSlider(min=0, max=100, step=1, value=4, description='Revisão:', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.inference_cost = widgets.FloatSlider(min=0, max=1, step=.005, value=.02, description='Inferência:', readout_format='.3f', continuous_update=False, layout=widgets.Layout(width='520px'))
        self.metrics_html = widgets.HTML()
        self.interpretation_html = widgets.HTML()
        self.cost_html = widgets.HTML()
        self.matrix_output = widgets.Output()
        self.snapshot_output = widgets.Output()
        self.best_f1_button = widgets.Button(description='🎯 Melhor F1')
        self.min_cost_button = widgets.Button(description='💰 Menor custo')
        self.snapshot_button = widgets.Button(description='📸 Snapshot estático')
        self.reset_button = widgets.Button(description='↺ Restaurar')
        self.scenario.observe(self._load_scenario, names='value')
        for control in self._controls(): control.observe(self._update, names='value')
        self.best_f1_button.on_click(lambda _: self._search_threshold('f1'))
        self.min_cost_button.on_click(lambda _: self._search_threshold('cost'))
        self.snapshot_button.on_click(self._snapshot)
        self.reset_button.on_click(lambda _: self._load_scenario({'new': self.scenario.value}))
        self._load_scenario({'new': self.scenario.value})

    def _controls(self):
        return [self.threshold, self.prevalence, self.separation, self.cost_fp, self.cost_fn, self.review_cost, self.inference_cost]

    def _load_scenario(self, change):
        cfg = SCENARIOS[change['new']]
        for c in self._controls(): c.unobserve(self._update, names='value')
        self.prevalence.value = cfg['prevalence']; self.separation.value = cfg['separation']; self.threshold.value = cfg['threshold']
        self.cost_fp.value = cfg['cost_fp']; self.cost_fn.value = cfg['cost_fn']; self.review_cost.value = cfg['review_cost']; self.inference_cost.value = cfg['inference_cost']
        for c in self._controls(): c.observe(self._update, names='value')
        self._update()

    def _state(self, threshold=None):
        threshold = self.threshold.value if threshold is None else threshold
        cm = simulate(self.prevalence.value, self.separation.value, threshold)
        metrics = metrics_from_cm(cm)
        cost = expected_cost(cm, self.cost_fp.value, self.cost_fn.value, self.review_cost.value, self.inference_cost.value)
        return cm, metrics, cost

    def _update(self, change=None):
        cm, metrics, cost = self._state()
        bars = ''.join(metric_bar(k, v) for k, v in metrics.items())
        self.metrics_html.value = f'<div style="padding:14px;border:1px solid #ddd;border-radius:12px"><h4>Indicadores — clique no nome para abrir o Glossário</h4>{bars}</div>'
        self.cost_html.value = f"<div style='padding:14px;border:1px solid #ddd;border-radius:12px;margin-top:10px'><b>Erros:</b> {glink('fp','FP')}={cm['FP']} · {glink('fn','FN')}={cm['FN']}<br><b>Custo esperado:</b> R$ {cost:,.2f}<br><b>Custo médio/decisão:</b> R$ {cost/N_CASES:,.4f}</div>"
        explanation = interpret(metrics, cm, self.prevalence.value, self.threshold.value, self.cost_fp.value, self.cost_fn.value, cost)
        self.interpretation_html.value = f'<div style="padding:14px;border:1px solid #ddd;border-radius:12px"><h4>Leitura do cenário</h4>{explanation}</div>'
        with self.matrix_output:
            clear_output(wait=True)
            matrix = np.array([[cm['TN'], cm['FP']], [cm['FN'], cm['TP']]])
            fig, ax = plt.subplots(figsize=(5.2, 4.0)); ax.imshow(matrix)
            ax.set_xticks([0,1], labels=['Negativo','Positivo']); ax.set_yticks([0,1], labels=['Negativo','Positivo'])
            ax.set_xlabel('Predito'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusão')
            for i in range(2):
                for j in range(2): ax.text(j, i, matrix[i,j], ha='center', va='center')
            plt.show(); plt.close(fig)

    def _search_threshold(self, objective):
        rows = []
        for t in np.linspace(.05,.95,91):
            cm, metrics, cost = self._state(float(t)); rows.append((float(t), metrics['F1'], cost))
        chosen = max(rows, key=lambda x: x[1]) if objective == 'f1' else min(rows, key=lambda x: x[2])
        self.threshold.value = round(chosen[0], 2)

    def _snapshot(self, _):
        cm, metrics, cost = self._state()
        with self.snapshot_output:
            clear_output(wait=True)
            display(pd.DataFrame({'Indicador': list(metrics) + ['FP','FN','Custo esperado'], 'Valor': list(metrics.values()) + [cm['FP'],cm['FN'],cost]}))

    def show(self):
        glossary_hint = widgets.HTML(f"<div style='padding:10px;border-left:4px solid #4a77b5;margin-bottom:10px'>📘 <b>Glossário em contexto:</b> ao mover os controles, clique nos nomes das métricas e nos links sugeridos pela interpretação. <a href='{GLOSSARY}' target='_blank'>Abrir glossário completo</a>.</div>")
        controls = widgets.VBox([glossary_hint, self.scenario, widgets.HTML(f"<b>Comportamento do classificador</b> · {glink('threshold','Threshold')} · {glink('support','Support')}"), self.threshold, self.prevalence, self.separation, widgets.HTML(f"<b>Custos do cenário</b> · {glink('fp','Falso positivo')} · {glink('fn','Falso negativo')}"), self.cost_fp, self.cost_fn, self.review_cost, self.inference_cost, widgets.HBox([self.best_f1_button,self.min_cost_button,self.reset_button]), self.snapshot_button])
        dashboard = widgets.HBox([widgets.VBox([self.metrics_html,self.cost_html], layout=widgets.Layout(width='48%')), widgets.VBox([self.interpretation_html,self.matrix_output], layout=widgets.Layout(width='50%'))], layout=widgets.Layout(width='100%', justify_content='space-between', align_items='flex-start'))
        display(controls); display(dashboard); display(self.snapshot_output)

lab = MetricScenarioLab()
lab.show()


## Missões de exploração — use o Glossário como ferramenta

**1 — [Threshold](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#threshold-de-decisão):** em Fraude bancária, mova de `0.70` para `0.30`. Antes de olhar a interpretação, tente prever o sentido de [Recall](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#recall), [Precisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#precisão), [FP](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-positivo), [FN](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-negativo), Acurácia e custo.

**2 — Contexto:** compare Triagem médica com Filtro de spam. Qual erro é menos tolerável em cada caso? Consulte [Falso positivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-positivo) e [Falso negativo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-negativo) antes de responder.

**3 — Melhor [F1](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#f1-score) ≠ menor custo:** use os botões **🎯 Melhor F1** e **💰 Menor custo** e compare os thresholds encontrados.

**4 — Prevalência:** reduza a prevalência mantendo os demais controles. Observe [Acurácia](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#acurácia), [Balanced Accuracy](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#balanced-accuracy) e [Support](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#support). Por que a acurácia pode ficar alta e ainda assim ser pouco informativa?

> **Regra das missões:** se uma mudança surpreender você, não tente decorar o resultado. Abra o conceito correspondente no Glossário, releia a definição e repita a simulação.


## 🧭 Mapa rápido: do sintoma ao conceito

Use este mapa durante a exploração:

```text
muitos FN              → Recall
muitos FP              → Precision / Specificity
classe rara            → Support / Balanced Accuracy
mover o corte          → Threshold
comparar FP e FN       → Matriz de confusão
equilibrar P e R       → F1-score
```

Depois de identificar o sintoma, volte às [palavras-chave do Glossário](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md) e formule a explicação com suas próprias palavras.


## Limites do simulador

Este é um **simulador didático**, não um estimador de desempenho real. Os scores são sintéticos, a variável Separação é uma abstração e os custos são exemplos. Thresholds reais exigem dados de validação representativos e probabilidades reais podem precisar de calibração.

```text
cenário → scores → threshold → decisões → matriz de confusão → métricas → custos → interpretação
```

> **O painel não existe para encontrar o maior número. Ele existe para treinar julgamento — e o Glossário existe para transformar surpresa em compreensão.**
